# Estimativa de Bandas SWIR — Treinamento de Modelos

Regressão espectral: **B04–B09 (SWIR)** a partir de **VNIR (B01, B02, B3N) + TIR (B10–B14)**.

| Modelo | Método | Hiperparâmetros |
|---|---|---|
| **PLS** | Partial Least Squares | `n_components` otimizado por CV |
| **XGBoost** | Gradient Boosting | `RandomizedSearchCV` + `KFold` |
| **GPR** | Gaussian Process | ARD-RBF (subamostrado, custo O(n³)) |

> **Entrada:** `data/samples/*.csv` (CSVs do Google Drive) — **Saídas:** `models/`

In [ ]:
# Instalar dependências (execute apenas uma vez)
# !pip install xgboost scikit-learn pandas seaborn joblib scipy matplotlib

import os, json, glob, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.model_selection import KFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from scipy.stats import uniform, randint

import xgboost as xgb

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')
print('Imports OK')

## 1. Parâmetros

Ajuste `CSV_DIR` para a pasta local com os CSVs baixados do Google Drive.

In [ ]:
CSV_DIR    = 'data/samples'    # pasta com CSVs do Drive
MODELS_DIR = Path('models')   # onde salvar modelos e métricas

N_JOBS     = -1   # todos os núcleos
N_CV       = 5    # KFold
N_ITER_XGB = 60   # iterações RandomizedSearchCV
GPR_NMAX   = 2000 # máx. amostras GPR
QUALITY_MIN = 0.5 # limiar quality (float, após conversão de escala)
SEED       = 42

FEATURES = ['B01', 'B02', 'B3N', 'B10', 'B11', 'B12', 'B13', 'B14']
TARGETS  = ['B04', 'B05', 'B06', 'B07', 'B08', 'B09']

# INT16 → unidade física
SCALE = {
    **dict.fromkeys(['B01','B02','B3N','B04','B05','B06','B07','B08','B09'], 1e-4),
    **dict.fromkeys(['B10','B11','B12','B13','B14'], 0.1),
    'quality': 1e-4,
}

MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f'CSV_DIR   : {CSV_DIR}')
print(f'MODELS_DIR: {MODELS_DIR.resolve()}')
print(f'Features  : {FEATURES}')
print(f'Alvos     : {TARGETS}')

## 2. Carregamento e Exploração dos Dados

In [ ]:
def carregar_dados():
    arquivos = sorted(glob.glob(os.path.join(CSV_DIR, '*.csv')))
    if not arquivos:
        raise FileNotFoundError(f'Nenhum CSV em {CSV_DIR!r}')
    print(f'Arquivos encontrados: {len(arquivos)}')
    for f in arquivos:
        print(f'  {Path(f).name}')

    df = pd.concat([pd.read_csv(f) for f in arquivos], ignore_index=True)
    print(f'\nRegistros brutos: {len(df):,}')

    for col, fator in SCALE.items():
        if col in df.columns:
            df[col] = df[col] * fator

    if 'quality' in df.columns:
        n_antes = len(df)
        df = df[df['quality'] >= QUALITY_MIN].copy()
        print(f'quality >= {QUALITY_MIN}: {n_antes:,} -> {len(df):,} registros')

    df = df[FEATURES + TARGETS].dropna()
    print(f'Registros finais   : {len(df):,}')
    return df

df_clean = carregar_dados()

X_all = df_clean[FEATURES].values.astype('float32')
Y_all = df_clean[TARGETS].values.astype('float32')
print(f'\nX shape: {X_all.shape}  |  Y shape: {Y_all.shape}')

In [ ]:
# Estatísticas descritivas
df_clean[FEATURES + TARGETS].describe().round(4).style.background_gradient(cmap='Blues', axis=0)

In [ ]:
# ── Heatmaps de correlação ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlação entre features
corr_ff = df_clean[FEATURES].corr()
sns.heatmap(corr_ff, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Correlação entre Features\n(VNIR + TIR)', fontsize=12)

# Correlação cruzada: features → SWIR
corr_ft = df_clean[FEATURES + TARGETS].corr().loc[FEATURES, TARGETS]
sns.heatmap(corr_ft, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[1], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Correlação Features → Alvos SWIR', fontsize=12)
axes[1].set_xlabel('Bandas SWIR (alvo)')
axes[1].set_ylabel('Features')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'correlacao.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Distribuição das bandas SWIR (alvos) ───────────────────────────────────
fig, axes = plt.subplots(1, 6, figsize=(16, 3), sharey=False)
for ax, banda in zip(axes, TARGETS):
    ax.hist(df_clean[banda], bins=60, color='#4C72B0', edgecolor='white', linewidth=0.3)
    ax.set_title(banda, fontsize=11)
    ax.set_xlabel('Reflectância')
    if ax == axes[0]:
        ax.set_ylabel('Contagem')
    ax.tick_params(labelsize=8)
plt.suptitle('Distribuição das Bandas SWIR (alvos)', y=1.03, fontsize=13)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'distribuicao_swir.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Split treino/teste + StandardScaler ─────────────────────────────────────
X_tr, X_te, Y_tr, Y_te = train_test_split(
    X_all, Y_all, test_size=0.2, random_state=SEED
)
print(f'Treino: {X_tr.shape[0]:,}  |  Teste: {X_te.shape[0]:,}')

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

kf = KFold(n_splits=N_CV, shuffle=True, random_state=SEED)

# Utilitários compartilhados
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def avaliar(y_true, y_pred):
    return {
        'r2'  : float(r2_score(y_true, y_pred)),
        'rmse': rmse(y_true, y_pred),
        'mae' : float(mean_absolute_error(y_true, y_pred)),
    }

def plot_scatter(preds_dict, Y_te_arr, titulo, cor):
    fig, axes = plt.subplots(2, 3, figsize=(14, 8))
    np.random.seed(SEED)
    for ax, banda in zip(axes.flat, TARGETS):
        j     = TARGETS.index(banda)
        y_t   = Y_te_arr[:, j]
        y_p   = preds_dict[banda]
        idx   = np.random.choice(len(y_t), min(6000, len(y_t)), replace=False)
        ax.scatter(y_t[idx], y_p[idx], alpha=0.25, s=4, color=cor)
        vmin, vmax = min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())
        ax.plot([vmin, vmax], [vmin, vmax], 'r--', lw=1.5)
        m = avaliar(y_t, y_p)
        ax.set_title(f'{banda}  R²={m["r2"]:.4f}  RMSE={m["rmse"]:.5f}', fontsize=9)
        ax.set_xlabel('Observado'); ax.set_ylabel('Estimado')
    plt.suptitle(titulo, fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

print('Setup OK')

## 3. PLS — Partial Least Squares

`n_components` ótimo encontrado por CV-RMSE (loop de 1 até `len(FEATURES)`).

In [ ]:
pls_models  = {}
pls_preds   = {}
pls_metrics = {}
pls_params  = {}
MAX_COMP    = len(FEATURES)

for j, banda in enumerate(TARGETS):
    y_tr_j = Y_tr[:, j]
    y_te_j = Y_te[:, j]

    # Busca n_components por CV-RMSE
    best_nc, best_cv = 1, float('inf')
    cv_curve = []
    for nc in range(1, MAX_COMP + 1):
        fold_rmses = []
        for tr_idx, val_idx in kf.split(X_tr_s):
            m = PLSRegression(n_components=nc)
            m.fit(X_tr_s[tr_idx], y_tr_j[tr_idx])
            fold_rmses.append(rmse(y_tr_j[val_idx], m.predict(X_tr_s[val_idx]).ravel()))
        media = float(np.mean(fold_rmses))
        cv_curve.append(media)
        if media < best_cv:
            best_cv, best_nc = media, nc

    modelo = PLSRegression(n_components=best_nc)
    modelo.fit(X_tr_s, y_tr_j)
    y_pred = modelo.predict(X_te_s).ravel()

    pls_models[banda]  = modelo
    pls_preds[banda]   = y_pred
    pls_metrics[banda] = avaliar(y_te_j, y_pred)
    pls_params[banda]  = {'n_components': best_nc, 'cv_rmse': best_cv}
    print(f'{banda}  n_components={best_nc}  CV-RMSE={best_cv:.5f}  R²={pls_metrics[banda]["r2"]:.4f}')

print('\nPLS concluído')

In [ ]:
# ── PLS: n_components escolhido por banda ──────────────────────────────────
pd.DataFrame([
    {'Banda': b, 'n_components': pls_params[b]['n_components'],
     'CV-RMSE': round(pls_params[b]['cv_rmse'], 6),
     'R² teste': round(pls_metrics[b]['r2'], 4),
     'RMSE teste': round(pls_metrics[b]['rmse'], 6)}
    for b in TARGETS
]).set_index('Banda').style.background_gradient(cmap='RdYlGn', subset=['R² teste'])

In [ ]:
plot_scatter(pls_preds, Y_te, 'PLS — Observado vs Estimado (conjunto de teste)', '#4C72B0')

## 4. XGBoost — Gradient Boosting

`RandomizedSearchCV` com `N_ITER_XGB` iterações e `KFold(N_CV)`. Score: neg_RMSE.

In [ ]:
XGB_PARAM_DIST = {
    'n_estimators':     randint(100, 700),
    'max_depth':        randint(3, 10),
    'learning_rate':    uniform(0.01, 0.29),
    'subsample':        uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'min_child_weight': randint(1, 10),
    'gamma':            uniform(0.0, 0.5),
    'reg_alpha':        uniform(0.0, 1.0),
    'reg_lambda':       uniform(0.5, 1.5),
}

xgb_models  = {}
xgb_preds   = {}
xgb_metrics = {}
xgb_params  = {}

for j, banda in enumerate(TARGETS):
    y_tr_j = Y_tr[:, j]
    y_te_j = Y_te[:, j]

    base = xgb.XGBRegressor(
        tree_method='hist', random_state=SEED, n_jobs=1, verbosity=0
    )
    search = RandomizedSearchCV(
        base, XGB_PARAM_DIST,
        n_iter=N_ITER_XGB, cv=kf,
        scoring='neg_root_mean_squared_error',
        random_state=SEED, n_jobs=N_JOBS,
        refit=True, verbose=0,
    )
    search.fit(X_tr_s, y_tr_j)

    melhor = search.best_estimator_
    y_pred = melhor.predict(X_te_s)

    xgb_models[banda]  = melhor
    xgb_preds[banda]   = y_pred
    xgb_metrics[banda] = avaliar(y_te_j, y_pred)
    xgb_params[banda]  = {**search.best_params_, 'cv_rmse': float(-search.best_score_)}
    print(f'{banda}  R²={xgb_metrics[banda]["r2"]:.4f}  RMSE={xgb_metrics[banda]["rmse"]:.5f}')

print('\nXGBoost concluído')

In [ ]:
# ── XGBoost: importância das features por banda ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, banda in zip(axes.flat, TARGETS):
    importancias = xgb_models[banda].feature_importances_
    idx_sorted   = np.argsort(importancias)[::-1]
    cores        = ['#2ecc71' if i < 3 else '#3498db' if i >= 5 else '#e74c3c'
                    for i in range(len(FEATURES))]
    ax.barh([FEATURES[i] for i in idx_sorted], importancias[idx_sorted],
            color=[cores[i] for i in idx_sorted], edgecolor='white')
    ax.set_title(f'{banda}', fontsize=10)
    ax.set_xlabel('Importância')
    ax.invert_yaxis()
plt.suptitle('XGBoost — Importância das Features por Banda SWIR', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'xgb_feature_importance.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── XGBoost: melhores hiperparâmetros ─────────────────────────────────────
pd.DataFrame(xgb_params).T.round(4).style.background_gradient(
    cmap='YlOrRd', subset=['cv_rmse']
)

In [ ]:
plot_scatter(xgb_preds, Y_te, 'XGBoost — Observado vs Estimado (conjunto de teste)', '#DD8452')

## 5. GPR — Gaussian Process Regression

Subamostrado a `GPR_NMAX` pontos (custo O(n³)). Kernel: **ARD-RBF** com comprimento de escala independente por feature + ruído branco.

In [ ]:
gpr_models  = {}
gpr_preds   = {}
gpr_metrics = {}

np.random.seed(SEED)
idx_gpr = np.random.choice(len(X_tr_s), min(GPR_NMAX, len(X_tr_s)), replace=False)
X_gpr_s = X_tr_s[idx_gpr]
print(f'Subamostrado: {len(X_gpr_s):,} pontos de treino para GPR')

kernel = (
    C(1.0, (1e-3, 1e3))
    * RBF(length_scale=np.ones(len(FEATURES)), length_scale_bounds=(1e-2, 1e2))
    + WhiteKernel(noise_level=1e-2, noise_level_bounds=(1e-5, 1.0))
)

for j, banda in enumerate(TARGETS):
    y_gpr = Y_tr[idx_gpr, j]
    modelo = GaussianProcessRegressor(
        kernel=kernel, alpha=0.0,
        normalize_y=True, n_restarts_optimizer=2, random_state=SEED,
    )
    modelo.fit(X_gpr_s, y_gpr)
    y_pred = modelo.predict(X_te_s)

    gpr_models[banda]  = modelo
    gpr_preds[banda]   = y_pred
    gpr_metrics[banda] = avaliar(Y_te[:, j], y_pred)
    print(f'{banda}  R²={gpr_metrics[banda]["r2"]:.4f}  RMSE={gpr_metrics[banda]["rmse"]:.5f}')

print('\nGPR concluído')

In [ ]:
plot_scatter(gpr_preds, Y_te, 'GPR — Observado vs Estimado (conjunto de teste)', '#55A868')

## 6. Comparação de Modelos

In [ ]:
all_metrics = {'PLS': pls_metrics, 'XGBoost': xgb_metrics, 'GPR': gpr_metrics}

# Tabela R² comparativa
df_r2 = pd.DataFrame(
    {mod: {b: all_metrics[mod][b]['r2'] for b in TARGETS} for mod in all_metrics}
)
df_r2.index.name = 'Banda'
print('=== R² por modelo e banda ===')
display(df_r2.round(4).style
    .background_gradient(cmap='RdYlGn', vmin=0, vmax=1)
    .format('{:.4f}')
    .set_caption('R² — quanto maior, melhor (máx = 1.0)'))

In [ ]:
# Tabela RMSE comparativa
df_rmse = pd.DataFrame(
    {mod: {b: all_metrics[mod][b]['rmse'] for b in TARGETS} for mod in all_metrics}
)
df_rmse.index.name = 'Banda'
print('=== RMSE por modelo e banda ===')
display(df_rmse.round(6).style
    .background_gradient(cmap='RdYlGn_r', axis=None)
    .format('{:.6f}')
    .set_caption('RMSE — quanto menor, melhor'))

In [ ]:
# ── Gráfico de barras agrupado: R² e RMSE ─────────────────────────────────
cores  = {'PLS': '#4C72B0', 'XGBoost': '#DD8452', 'GPR': '#55A868'}
x      = np.arange(len(TARGETS))
width  = 0.26

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for i, (mod, cor) in enumerate(cores.items()):
    r2s   = [all_metrics[mod][b]['r2']   for b in TARGETS]
    rmses = [all_metrics[mod][b]['rmse'] for b in TARGETS]
    ax1.bar(x + i * width, r2s,   width, label=mod, color=cor,
            edgecolor='black', linewidth=0.5, alpha=0.85)
    ax2.bar(x + i * width, rmses, width, label=mod, color=cor,
            edgecolor='black', linewidth=0.5, alpha=0.85)

ax1.set_xticks(x + width); ax1.set_xticklabels(TARGETS)
ax1.set_ylabel('R²'); ax1.set_title('R² por Modelo e Banda SWIR')
ax1.set_ylim(0, 1.05)
ax1.axhline(0.9, color='red', ls='--', lw=1, alpha=0.6, label='R²=0.9')
ax1.legend(); ax1.grid(axis='y', alpha=0.3)

ax2.set_xticks(x + width); ax2.set_xticklabels(TARGETS)
ax2.set_ylabel('RMSE'); ax2.set_title('RMSE por Modelo e Banda SWIR')
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'comparacao_modelos.png', bbox_inches='tight')
plt.show()

## 7. Salvar Modelos e Artefatos

In [ ]:
# Scaler
joblib.dump(scaler, MODELS_DIR / 'scaler.joblib')

# Modelos por banda
for banda in TARGETS:
    joblib.dump(pls_models[banda], MODELS_DIR / f'pls_{banda}.joblib')
    joblib.dump(xgb_models[banda], MODELS_DIR / f'xgb_{banda}.joblib')
    joblib.dump(gpr_models[banda], MODELS_DIR / f'gpr_{banda}.joblib')

# Parâmetros
best_params_out = {'pls': pls_params, 'xgb': xgb_params}
with open(MODELS_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params_out, f, indent=2, default=str)

# Métricas
metrics_out = {mod.lower(): {b: v for b, v in vals.items()}
               for mod, vals in all_metrics.items()}
with open(MODELS_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)

print('Arquivos salvos em', MODELS_DIR.resolve())
for f in sorted(MODELS_DIR.iterdir()):
    print(f'  {f.name}')